# Phase 2: executed frozen-evidence review

Ruganji Prince — 04-652 Artificial Intelligence System Design

This notebook reads approved CSV/JSON artifacts. No training, tuning, threshold selection or new test prediction occurs. The historical test partition was previously exposed; results are not from a pristine test set. Class 4 is environmental shade, not hardware failure.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/raw/pv_fault_dataset.csv').is_file())
E = ROOT/'phase2_implementation/stage3/evidence'
def read(name):
    return json.loads((E/name).read_text(encoding='utf-8-sig'))
comparison = pd.read_csv(E/'final_candidate_comparison.csv')
print('Population: TEST [1227337,1373798), 146461 observations; no irradiance filtering.')
print('Model decisions frozen from TRAIN / VALIDATION; no fitting in this notebook.')


Population: TEST [1227337,1373798), 146461 observations; no irradiance filtering.
Model decisions frozen from TRAIN / VALIDATION; no fitting in this notebook.


## Frozen protocol and threshold chronology

In [2]:
thresholds = read('stage3_threshold_selection.json')
receipt = read('threshold_freeze_receipt.json')
event = read('test_access_started.json')
threshold_hash = hashlib.sha256((E/'stage3_threshold_selection.json').read_bytes()).hexdigest()
assert receipt['threshold_sha256'] == event['threshold_sha256'] == threshold_hash
assert thresholds['frozen_utc'] <= receipt['created_utc'] < event['created_utc']
print(pd.DataFrame([{k:r[k] for k in ['candidate','threshold','recall','precision','f1','normal_false_alarm_rate','shade_false_alarm_rate','combined_nonhardware_FPR']} for r in thresholds['results'].values()]).to_string(index=False))
print('Threshold freeze before revised test access: verified.')


   candidate  threshold  recall  precision       f1  normal_false_alarm_rate  shade_false_alarm_rate  combined_nonhardware_FPR
      LR__F6   0.419382     1.0   0.962244 0.980759                 0.001419                0.012671                  0.004694
      RF__E4   0.591356     1.0   0.976464 0.988092                 0.000898                0.007718                  0.002883
LightGBM__E4   0.138600     1.0   0.976999 0.988366                 0.000378                0.008755                  0.002816
Threshold freeze before revised test access: verified.


## Final TEST classification and hardware-class recall

In [3]:
print(comparison[['candidate','test_macro_f1','test_weighted_f1','test_macro_AP','test_R1','test_R2','test_R3','test_hardware_macro_recall','same_family_test_delta_s']].to_string(index=False))
classes = pd.read_csv(E/'final_test_class_metrics.csv')
print('\nComplete per-class TEST metrics:')
print(classes.to_string(index=False))


   candidate  test_macro_f1  test_weighted_f1  test_macro_AP  test_R1  test_R2  test_R3  test_hardware_macro_recall  same_family_test_delta_s
      LR__F6       0.898389          0.940280       0.917423 1.000000 0.997797 0.851867                    0.949888                       NaN
      RF__E4       0.950481          0.957199       0.976219 0.999576 0.990749 1.000000                    0.996775                 -0.006291
LightGBM__E4       0.902137          0.987196       0.993578 0.499365 0.992952 0.999170                    0.830495                 -0.002549

Complete per-class TEST metrics:
   candidate  class  precision   recall       f1  support       AP
      LR__F6      0   0.971218 0.961546 0.966358   124512 0.996409
      LR__F6      1   0.853271 1.000000 0.920827     2361 0.968998
      LR__F6      2   0.925245 0.997797 0.960153     2270 0.974387
      LR__F6      3   0.996602 0.851867 0.918568     2410 0.987691
      LR__F6      4   0.703092 0.750537 0.726040    14908 0.659

## Frozen hardware alerts — separate from argmax classification

In [4]:
print(pd.read_csv(E/'final_hardware_alert_metrics.csv').to_string(index=False))
print('The 1% target applies to combined normal/shade FPR; separate FPRs remain visible.')


   candidate  threshold  precision   recall       f1  normal_false_alarm_rate  shade_false_alarm_rate  combined_nonhardware_FPR   TP  FP  FN  coverage  average_precision
      LR__F6   0.419382   0.957829 1.000000 0.978460                 0.001084                0.011739                  0.002223 7041 310   0       1.0           0.989638
      RF__E4   0.591356   0.983719 0.995455 0.989552                 0.000161                0.006439                  0.000832 7009 116  32       1.0           0.993354
LightGBM__E4   0.138600   0.983604 0.996875 0.990195                 0.000088                0.007110                  0.000839 7019 117  22       1.0           0.994775
The 1% target applies to combined normal/shade FPR; separate FPRs remain visible.


## Resource and robustness evidence

Timing and simulated dropout use VALIDATION block 13. Artifact sizes describe complete TRAIN-fitted serialized pipelines.

In [5]:
print(comparison[['candidate','physical_channels','mathematical_features','validation_p50_ms','validation_p95_ms','artifact_MiB','validation_dropout_loss']].to_string(index=False))
criteria = pd.read_csv(E/'final_engineering_criteria.csv', keep_default_na=False)
print('\nEngineering assessment:')
print(criteria.pivot(index='criterion',columns='candidate',values='status').to_string())
assert not comparison.passes_all_applicable.any()
print('No evaluated configuration satisfied all predeclared engineering criteria.')


   candidate  physical_channels  mathematical_features  validation_p50_ms  validation_p95_ms  artifact_MiB  validation_dropout_loss
      LR__F6                  6                     12           12.79435          14.716925      0.003742                 0.014986
      RF__E4                  4                      8           31.32045          36.808295     11.665363                 0.008163
LightGBM__E4                  4                     13            3.79575           6.124760      1.569197                 0.010639

Engineering assessment:
candidate                                  LR__F6 LightGBM__E4 RF__E4
criterion                                                            
artifact_le_10MiB                            PASS         PASS   FAIL
each_hardware_classification_recall_ge_090   FAIL         FAIL   PASS
macro_f1_ge_080                              PASS         PASS   PASS
same_family_test_delta_le_005                 N/A         PASS   PASS
sparse_channels_le_4       

## Explanations — balanced VALIDATION sample, non-causal

In [6]:
metadata = read('explainability_metadata.json')
print('Sample size:',metadata['sample_size'],'class counts:',metadata['class_counts'],'seed:',metadata['seed'])
print('Balanced sample does not represent class prevalence.')
permutation = pd.read_csv(E/'physical_channel_permutation.csv')
print(permutation.groupby(['candidate','channel']).macro_f1_loss.mean().to_string())
print('Dependent engineered features are regenerated after raw-channel permutation.')
print('Tree SHAP maximum raw-margin additivity error:',metadata['SHAP']['additivity_max_absolute_error'])


Sample size: 1000 class counts: {'0': 200, '1': 200, '2': 200, '3': 200, '4': 200} seed: 42
Balanced sample does not represent class prevalence.
candidate     channel
LR__F6        idc1       0.386699
              idc2       0.261754
              irr        0.256463
              pvt        0.024552
              vdc1       0.095354
              vdc2       0.625207
LightGBM__E4  idc1       0.295352
              idc2       0.247032
              vdc1       0.088445
              vdc2       0.434170
RF__E4        idc1       0.229538
              idc2       0.288719
              vdc1       0.009023
              vdc2       0.519344
Dependent engineered features are regenerated after raw-channel permutation.
Tree SHAP maximum raw-margin additivity error: 6.128431095930864e-14


## Selected diagnostics

![TEST per-class performance](../stage4/figures/test_per_class_metrics.png)

![TEST hardware alerts, thresholds from VALIDATION](../stage4/figures/test_hardware_alerts.png)

![VALIDATION physical-channel importance](../stage4/figures/validation_physical_channel_importance.png)

![TEST performance; VALIDATION timing; TRAIN-fitted artifacts](../stage4/figures/performance_resource_tradeoff.png)

## Integrity and interpretation limits

In [7]:
raw = ROOT/'data/raw/pv_fault_dataset.csv'
h = hashlib.sha256()
with raw.open('rb') as f:
    for chunk in iter(lambda:f.read(1048576), b''):
        h.update(chunk)
assert h.hexdigest() == '717278fdc338bf6713293c0495421a8be9ea02a0b64144e86e957a531edecb8a'
print('Immutable raw CSV SHA-256:',h.hexdigest())
print('No timestamps/source-index predictors; no global minimum-sensing or causal claim.')
print('Prior legacy test exposure is disclosed. Full limitations appear in the final report.')


Immutable raw CSV SHA-256: 717278fdc338bf6713293c0495421a8be9ea02a0b64144e86e957a531edecb8a
No timestamps/source-index predictors; no global minimum-sensing or causal claim.
Prior legacy test exposure is disclosed. Full limitations appear in the final report.
